# Setup

In [1]:
import json
from encodings import charmap

from torch.cuda import is_available as is_cuda_available
import numpy as np
from lighthouse.models import CGDETRPredictor
from utils.helper_methods import split_video_ffmpeg, parse_timestamps, extract_clips, adjust_video_speed
import pandas as pd
import os

C:\Users\denve\miniconda3\envs\moment-retrieval-and-video-summarization\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model weights can be downloaded from https://zenodo.org/records/13960580
Checkpoint file should correspond to the feature selected in predictor

For slowfast video encoding features additional weights should be downloaded. They can be found in lighthouse GitHub page (https://github.com/line/lighthouse/tree/main) or downloaded from [here](https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl)

In [2]:
device = "cuda" if is_cuda_available() else "cpu"
device

'cuda'

In [3]:
# model = CGDETRPredictor('model_weights/clip_cg_detr_qvhighlight.ckpt', device=device,
#                         feature_name='clip')

model = CGDETRPredictor('model_weights/clip_slowfast_cg_detr_qvhighlight.ckpt', device=device,
                        feature_name='clip_slowfast', slowfast_path="model_weights/SLOWFAST_8x8_R50.pkl", pann_path=None)

# Video preparation
CG-DETR model has a major limitation -- it doesn't allow videos longer than 150 seconds. We will try several approaches to this problem

In [4]:
# # Load events which should be retrieved
# events = pd.read_csv("Annotations/7-TVSum-video_25-i6324060.txt")[['annotation', 'timestamp']]
# events['timestamp'].replace(np.nan, None)
# events['timestamp'] = events['timestamp'].map(parse_timestamps)

In [5]:
# set the original video path
video_name = "video_27"
og_video_path = f"TVSum/{video_name}.mp4"

In [6]:
with open(f"Descriptions/{video_name}.json", 'r', encoding='utf-8') as f:
    events_json = json.load(f)

events_full = pd.json_normalize(events_json, record_path=['llm_results'])[['prompt_type', 'vlm_source', 'vlm_frame_count', 'parsed_events']]

events_full = events_full[(events_full['prompt_type'] == 'concise') & (events_full['vlm_frame_count'] == 50) & (events_full['vlm_source'] == 'long')] # we only select the most optimized parameters from the vlm/llm pipeline

events = events_full['parsed_events'].to_frame()
events['timestamps'] = np.nan

In [9]:
events_full

,prompt_type,vlm_source,vlm_frame_count,parsed_events
16,concise,long,50,[The video starts with an introduction screen ...


## Splitting
The most obvious way is splitting videos in chunks. We cut the original video in segments no longer than 150 seconds and then process them seperately

In [7]:
output_path = f"processed_videos/{video_name}_chunks"
split_video_ffmpeg(og_video_path, output_path, 150)

Splitting TVSum/video_27.mp4 into 150s chunks...
Chunks are successfully saved into 'processed_videos/video_27_chunks' folder.


'processed_videos/video_27_chunks'

In [8]:
results = []
for row in events.iterrows():
    results_per_entry = [] # results for every event list

    queries, true_timestamps = row[1]
    for query in queries:
        results_per_chunk = [] # results for every query

        print(f"Query: {query}")
        print(f"True timestamp: {true_timestamps}")

        for index, chunk in enumerate(os.listdir(output_path)):
            if chunk.endswith('.mp4'):
                video_path = output_path + '/' + chunk
                video = model.encode_video(video_path)
                prediction = model.predict(query, video)
                results_per_chunk.append((index, prediction))

                print("\tChunk:", index)
                print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0]}, {prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        print('-' * 50)

        results_per_entry.append(results_per_chunk)

    results.append(results_per_entry)

Query: The video starts with an introduction screen displaying "MHD" in white letters against a black background.
True timestamp: nan
	Chunk: 0
		Closest Moment: 0.0202, 10.49, Corresponding Confidence score: 0.9951
	Chunk: 1
		Closest Moment: 129.5547, 143.9304, Corresponding Confidence score: 0.9783
	Chunk: 2
		Closest Moment: 59.2291, 64.0, Corresponding Confidence score: 0.999
--------------------------------------------------
Query: Students are shown participating in various activities such as walking down a path lined with green leaves and wearing garlands made of greenery.
True timestamp: nan
	Chunk: 0
		Closest Moment: 110.6975, 123.7276, Corresponding Confidence score: 0.9953
	Chunk: 1
		Closest Moment: 74.9687, 86.0196, Corresponding Confidence score: 0.9972
	Chunk: 2
		Closest Moment: 50.5507, 59.1454, Corresponding Confidence score: 0.9996
--------------------------------------------------
Query: The students celebrate together outdoors under trees or buildings, waving fla

RuntimeError: Input The students express happiness, show excitement, have fun, socialize happily, speak directly to camera, listen attentively, react positively, move energetically, create vibrant colors, reflect sunlight, feel joyful, experience camaraderie, commemorate special occasions, highlight unity, emphasize community spirit, convey pride, demonstrate resilience, foster friendship, promote school culture, encourage participation, instill enthusiasm, build connections, preserve traditions, nurture friendships, support academic achievements, honor heritage, celebrate diversity, recognize excellence, acknowledge contributions, appreciate teamwork, value individuality, respect differences, cherish shared experiences, maintain continuity, sustain legacy, cultivate growth, enhance relationships, strengthen bonds, enrich lives. is too long for context length 77

In [ ]:

# for index, query_result in enumerate(chunked_results):
#     print(f"Query: {query_result['query']}")
#     print(f"True timestamp: {query_result['true_timestamp']}")
#
#
#     for chunk in query_result['predictions']:
#         print("\tChunk:", chunk[0])
#         print(f"\t\tClosest Moment: {chunk[1]['pred_relevant_windows'][0][0], chunk[1]['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {chunk[1]['pred_relevant_windows'][0][2]}")
#     print("-" * 50)

Now we have a problem, because we get a timestamp for each chunk, but we only need one timestamp for the whole video. The simplest solution is to pick confidence score, although this approach has a lot of shortcomings, we will use it, as alternatives require more complex algorithms

In [ ]:
chunks_timestamp_results = []
for results_per_entry in results:

    max_results_per_entry = []
    for results_per_chunk in results_per_entry:
        max_score = 0.0
        best_timestamp = ()
        for chunk in results_per_chunk:
            prediction = chunk[1]
            if prediction['pred_relevant_windows'][0][2] > max_score:
                max_score = prediction['pred_relevant_windows'][0][2]
                best_timestamp = (prediction['pred_relevant_windows'][0][0] + 150 * chunk[0], prediction['pred_relevant_windows'][0][1] + 150 * chunk[0]) # we add 150 multiplied by chunk index to get the global timestamp(the model returns timestamp local to a particular chunk)
        max_results_per_entry.append(best_timestamp)

    chunks_timestamp_results.append(max_results_per_entry)

chunks_timestamp_results

## Speeding up
Another approach is speeding up the video until it becomes short enough

In [ ]:
output_path = f"processed_videos/{video_name}_sped_up.mp4"
speed_factor = adjust_video_speed(og_video_path, output_path, 150)

In [ ]:
sped_up_timestamps_results = []

for row in events.iterrows():
    queries, timestamps = row[1]
    predicted_timestamps = []


    results_per_entry = []
    for query in queries:
        print(f"Query: {query}")
        print(f"True timestamp: {query}")

        video = model.encode_video(output_path)
        prediction = model.predict(query, video)

        print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        results_per_entry.append((prediction['pred_relevant_windows'][0][0] * speed_factor, prediction['pred_relevant_windows'][0][1] * speed_factor))
        print("-" * 50)

    sped_up_timestamps_results.append(results_per_entry)

In [ ]:
# for results_per_entry in results:
#     for result_per_event in results_per_entry:
#
#     prediction = query_result['predictions']


In [ ]:
chunks_timestamp_results

In [ ]:
events_full['predicted_timestamps_chunks'] = chunks_timestamp_results
events_full['predicted_timestamps_sped_up'] = sped_up_timestamps_results

In [ ]:
events_full.to_json(f"Timestamps/{og_video_path.split('/')[-1].split('.')[0]}_with_predictions.json", orient='records')